In [33]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV

import xgboost as xgb

import warnings
warnings.filterwarnings("ignore")

In [34]:
df = pd.read_csv("../data/processed/features_v2.csv")

df['date'] = pd.to_datetime(df['date'])

df = df.sort_values('date')

In [35]:
split_date = df['date'].quantile(0.8)

train = df[df['date'] < split_date].copy()
test  = df[df['date'] >= split_date].copy()

In [36]:
# Compute historical circuit average (train data only to avoid leakage)

circuit_avg = train.groupby("circuitId")["avgLapTime_s"].mean()

# Map to both train and test
train["circuit_avg_lap"] = train["circuitId"].map(circuit_avg)
test["circuit_avg_lap"]  = test["circuitId"].map(circuit_avg)

In [37]:
global_avg = train["avgLapTime_s"].mean()

test["circuit_avg_lap"] = test["circuit_avg_lap"].fillna(global_avg)

In [38]:
train["lap_time_relative"] = train["avgLapTime_s"] - train["circuit_avg_lap"]
test["lap_time_relative"]  = test["avgLapTime_s"] - test["circuit_avg_lap"]

In [39]:
target_finish = "positionOrder"
target_lap    = "avgLapTime_s"
target_points = "points"

drop_cols = [
    "raceId", "driverId", "constructorId",
    "date"
]

features = [col for col in df.columns 
            if col not in drop_cols + [target_finish, target_lap, target_points]]

In [40]:
X_train = train[features]
X_test  = test[features]

y_train = train["lap_time_relative"]
y_test  = test["lap_time_relative"]

In [41]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestRegressor(max_depth=10, min_samples_split=5, n_estimators=300,
                      n_jobs=-1, random_state=42)

In [43]:
rf_preds = rf.predict(X_test)

rf_mae  = mean_absolute_error(y_test, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_r2   = r2_score(y_test, rf_preds)

print("Random Forest Results")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)
print("R2:", rf_r2)

Random Forest Results
MAE: 11.305785052854459
RMSE: 26.390020437687017
R2: 0.029879673223025427


In [44]:
xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror"
)

xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=300, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [45]:
xgb_preds = xgb_model.predict(X_test)

xgb_mae  = mean_absolute_error(y_test, xgb_preds)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
xgb_r2   = r2_score(y_test, xgb_preds)

print("XGBoost Results")
print("MAE:", xgb_mae)
print("RMSE:", xgb_rmse)
print("R2:", xgb_r2)

XGBoost Results
MAE: 11.510653296103303
RMSE: 26.80272037242304
R2: -0.0007000056072437033


In [32]:
results = pd.DataFrame({
    "model": ["RandomForest_v2", "XGBoost_v2"],
    "target": ["AvgLapTime", "AvgLapTime"],
    "MAE": [rf_mae, xgb_mae],
    "RMSE": [rf_rmse, xgb_rmse],
    "R2": [rf_r2, xgb_r2]
})

results

,model,target,MAE,RMSE,R2
0,RandomForest_v2,AvgLapTime,11.305785,26.39002,0.02988
1,XGBoost_v2,AvgLapTime,11.510653,26.80272,-0.00070


📘 Lap Time Modeling – Normalization Experiment

Motivation:
Initial models predicting raw average lap time achieved:
*R² ≈ 0.15
*MAE ≈ 12 seconds

Although modest, the performance was consistent across Random Forest and XGBoost.

However, raw lap time is heavily influenced by structural circuit differences (e.g., Monaco vs Spa), which may dominate variance in the dataset.

To reduce this structural bias, a normalization approach was tested.

Method: Circuit-Relative Normalization

To isolate driver and constructor performance from circuit-specific effects, lap times were normalized relative to the historical circuit mean.

The procedure was:

1.Split the dataset chronologically (time-aware split).
2.Compute average lap time per circuit using training data only.
3.Create a new target: lap_time_relative = avgLapTime_s − circuit_average

This removes track-length effects and focuses the model on performance deviations within each circuit.

Importantly, circuit averages were computed only from the training set to avoid data leakage.

Results of Normalization:

After retraining Random Forest and XGBoost on the normalized target:
-R² dropped to approximately 0.00
-Model performance became comparable to predicting the mean

Interpretation
The decline in R² suggests:

1.A large portion of predictive signal in the original model was derived from structural circuit differences.
2.After removing circuit-level variance, remaining performance variation was largely influenced by:
-Weather conditions
-Tyre strategy
-Safety cars
-Track evolution
-Fuel load
-Mechanical failures

These variables are not present in the dataset.

Thus, the model lacks sufficient explanatory features to predict intra-circuit lap time deviations effectively.

Decision
Given that normalization:
-Reduced predictive performance significantly
-Removed meaningful structural signal
-Did not introduce new explanatory variables

The original raw lap time model (R² ≈ 0.15) was retained.

Although modest, it:

-Reflects realistic predictive capability
-Uses available structured features
-Avoids over-processing the target variable

“The modest predictive performance for lap time (R² ≈ 0.15) reflects the absence of dynamic race-day variables such as weather conditions, tyre compound selection, safety car periods, fuel load variation, and track evolution. These contextual factors significantly influence lap performance but are not captured in structured historical race datasets. Future work incorporating telemetry-level or session-level data would likely improve model explanatory power.”